In [1]:
import sys
sys.path.append(r"C:\Users\zhaoh\Desktop\CDS")
import os 
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader
from src.utils import load_video_data, evaluate, extract_faces_preserve_structure_mtcnn
from src.data.image.data_loader import ImageDataset
from sklearn.utils.class_weight import compute_class_weight
from transformers import ViTForImageClassification


c:\Users\zhaoh\Desktop\CDS\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Step 1: Get video folders and aligned labels
video_dirs, labels, emotion_to_idx = load_video_data(
    metadata_csv="C:/Users/zhaoh/Desktop/processed_train/metadata_extracted.csv",
    frames_root_dir="C:/Users/zhaoh/Desktop/processed_train/frames_test/"
)

transform = transforms.Compose([
    # Resize to model input
    transforms.Resize((224, 224)),

    # Data augmentation
    transforms.RandomHorizontalFlip(p=0.5),  # flip left-right
    transforms.RandomRotation(10),           # rotate ±10 degrees
    transforms.ColorJitter(
        brightness=0.2, 
        contrast=0.2, 
        saturation=0.2, 
        hue=0.05
    ),  # adjust colors slightly

    # Convert to tensor
    transforms.ToTensor(),

    # Optional: normalize with ImageNet stats if using pretrained ResNet
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    video_dirs=video_dirs,
    labels=labels,
    transform=transform,
    n_frames=4
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
)

labels_np = np.array(labels)

# compute weights automatically
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels_np),
    y=labels_np
)

# convert to tensor
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
class_weights = class_weights / class_weights.mean()

class_weights = class_weights.to(device)

print("Normalized class weights:", class_weights)


Using device: cuda
Normalized class weights: tensor([0.5623, 2.2254, 2.3230, 0.3525, 0.1301, 0.8984, 0.5083],
       device='cuda:0')


In [3]:
print(emotion_to_idx)

{'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'neutral': 4, 'sadness': 5, 'surprise': 6}


In [4]:
# Step 1: Get video folders and aligned labels
test_video_dirs, test_labels, test_emotion_to_idx = load_video_data(
    metadata_csv="C:/Users/zhaoh/Desktop/processed_test/metadata_extracted.csv",
    frames_root_dir="C:/Users/zhaoh/Desktop/processed_test/frames_test/"
)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

test = ImageDataset(
    video_dirs=test_video_dirs,
    labels=test_labels,
    transform=transform,
    n_frames=4
)

test_dataloader = DataLoader(
    test,
    batch_size=8,
    shuffle=False,
    num_workers=2,
)
assert emotion_to_idx == test_emotion_to_idx, "Emotion-to-index mappings differ between train and test!"

In [5]:
# Load Pre-trained model on FER2013 Dataset

model = ViTForImageClassification.from_pretrained(
    "afurkank/vit-face-expression",
    num_labels=7
)

model = model.to(device)

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 976.03it/s, Materializing param=vit.layernorm.weight]                                 


In [6]:
# criterion = nn.CrossEntropyLoss(weight=class_weights)

# # -------------------------------
# # 4. Optimizer
# # -------------------------------
# optimizer = optim.AdamW(model.parameters(), lr=1e-5)  # small LR for pretrained model

# # -------------------------------
# # 5. Training loop
# # -------------------------------
# num_epochs = 10

# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0
#     total_correct = 0
#     total_samples = 0

#     for pixel_values, labels in train_dataloader:
#         # pixel_values: [B, T, C, H, W]
#         B, T, C, H, W = pixel_values.shape

#         pixel_values = pixel_values.view(B * T, C, H, W).to(device)
#         labels = labels.to(device)

#         optimizer.zero_grad()

#         outputs = model(pixel_values=pixel_values)
#         logits = outputs.logits

#         logits = logits.view(B, T, -1)     # [B, T, num_classes]
#         logits = logits.mean(dim=1)

#         loss = criterion(logits, labels)
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item() * labels.size(0)
#         preds = logits.argmax(dim=1)
#         total_correct += (preds == labels).sum().item()
#         total_samples += labels.size(0)

#     avg_loss = total_loss / total_samples
#     train_acc = total_correct / total_samples
#     print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}")

#     # -------------------------------
#     # 6. Evaluation on test set
#     # -------------------------------
#     model.eval()
#     total_correct = 0
#     total_samples = 0
#     with torch.no_grad():
#         for pixel_values, labels in test_dataloader:
#             B, T, C, H, W = pixel_values.shape

#             pixel_values = pixel_values.view(B * T, C, H, W).to(device)
#             labels = labels.to(device)

#             outputs = model(pixel_values=pixel_values)
#             logits = outputs.logits

#             logits = logits.view(B, T, -1).mean(dim=1)

#             preds = logits.argmax(dim=1)
#             total_correct += (preds == labels).sum().item()
#             total_samples += labels.size(0)

#     test_acc = total_correct / total_samples
#     print(f"Epoch {epoch+1}/{num_epochs} | Test Acc: {test_acc:.4f}")

In [7]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

# -------------------------------
# 4. Optimizer
# -------------------------------
optimizer = optim.AdamW(model.parameters(), lr=1e-5)  # small LR for pretrained model

# -------------------------------
# 5. Training loop
# -------------------------------
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    for pixel_values, labels in train_dataloader:
        # pixel_values: [B, T, C, H, W]
        B, T, C, H, W = pixel_values.shape

        pixel_values = pixel_values.view(B * T, C, H, W).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values=pixel_values)
        logits = outputs.logits

        logits = logits.view(B, T, -1)  # [B, T, num_classes]

        # -------------------------------
        # Pick most confident frame per video
        # -------------------------------
        frame_confidence, _ = logits.max(dim=2)      # max logit per frame: [B, T]
        best_frame_idx = frame_confidence.argmax(dim=1)  # index of best frame per video: [B]
        logits = logits[torch.arange(B), best_frame_idx]  # logits of best frame: [B, num_classes]

        # -------------------------------
        # Loss and backward
        # -------------------------------
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    train_acc = total_correct / total_samples
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}")

    # -------------------------------
    # 6. Evaluation on test set
    # -------------------------------
    model.eval()
    total_correct = 0
    total_samples = 0
    with torch.no_grad():
        for pixel_values, labels in test_dataloader:
            B, T, C, H, W = pixel_values.shape

            pixel_values = pixel_values.view(B * T, C, H, W).to(device)
            labels = labels.to(device)

            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits

            logits = logits.view(B, T, -1)  # [B, T, num_classes]

            # -------------------------------
            # Pick most confident frame per video
            # -------------------------------
            frame_confidence, _ = logits.max(dim=2)
            best_frame_idx = frame_confidence.argmax(dim=1)
            logits = logits[torch.arange(B), best_frame_idx]

            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    test_acc = total_correct / total_samples
    print(f"Epoch {epoch+1}/{num_epochs} | Test Acc: {test_acc:.4f}")

Epoch 1/10 | Train Loss: 2.0148 | Train Acc: 0.2744
Epoch 1/10 | Test Acc: 0.2651
Epoch 2/10 | Train Loss: 1.9157 | Train Acc: 0.3247
Epoch 2/10 | Test Acc: 0.2619
Epoch 3/10 | Train Loss: 1.8974 | Train Acc: 0.3038
Epoch 3/10 | Test Acc: 0.2858
Epoch 4/10 | Train Loss: 1.8699 | Train Acc: 0.2910
Epoch 4/10 | Test Acc: 0.2675
Epoch 5/10 | Train Loss: 1.8417 | Train Acc: 0.3194
Epoch 5/10 | Test Acc: 0.3153
Epoch 6/10 | Train Loss: 1.7974 | Train Acc: 0.3075
Epoch 6/10 | Test Acc: 0.2675
Epoch 7/10 | Train Loss: 1.7085 | Train Acc: 0.3221
Epoch 7/10 | Test Acc: 0.3288
Epoch 8/10 | Train Loss: 1.6300 | Train Acc: 0.3449
Epoch 8/10 | Test Acc: 0.2922
Epoch 9/10 | Train Loss: 1.5318 | Train Acc: 0.3762
Epoch 9/10 | Test Acc: 0.2381
Epoch 10/10 | Train Loss: 1.4397 | Train Acc: 0.3965
Epoch 10/10 | Test Acc: 0.2683


In [8]:
# Save model state_dict and optimizer state_dict
torch.save({
    'epoch': 10,                  # save current epoch number
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': avg_loss,
}, "vit_fer_checkpoint_conf.pth")

print("Model checkpoint saved.")

Model checkpoint saved.


In [9]:
# # Load checkpoint
# checkpoint = torch.load("vit_fer_checkpoint.pth", map_location=device)

# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# start_epoch = checkpoint['epoch'] + 1  # continue from next epoch

# print(f"Resuming training from epoch {start_epoch}")

In [10]:
# num_epochs = start_epoch + 5  # train 5 more epochs

# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0
#     total_correct = 0
#     total_samples = 0

#     for pixel_values, labels in train_dataloader:
#         # pixel_values: [B, T, C, H, W]
#         B, T, C, H, W = pixel_values.shape

#         pixel_values = pixel_values.view(B * T, C, H, W).to(device)
#         labels = labels.to(device)

#         optimizer.zero_grad()

#         outputs = model(pixel_values=pixel_values)
#         logits = outputs.logits

#         logits = logits.view(B, T, -1)     # [B, T, num_classes]
#         logits = logits.mean(dim=1)

#         loss = criterion(logits, labels)
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item() * labels.size(0)
#         preds = logits.argmax(dim=1)
#         total_correct += (preds == labels).sum().item()
#         total_samples += labels.size(0)

#     avg_loss = total_loss / total_samples
#     train_acc = total_correct / total_samples
#     print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}")

In [11]:
# model.eval()  # important: disables dropout, batchnorm updates
# total_correct = 0
# total_samples = 0

# all_preds = []
# all_labels = []

# with torch.no_grad():  # no gradients needed for evaluation
#     for pixel_values, labels in test_dataloader:
#         B, T, C, H, W = pixel_values.shape

#         pixel_values = pixel_values.view(B * T, C, H, W).to(device)
#         labels = labels.to(device)

#         outputs = model(pixel_values=pixel_values)
#         logits = outputs.logits

#         logits = logits.view(B, T, -1).mean(dim=1)

#         preds = logits.argmax(dim=1)
#         total_correct += (preds == labels).sum().item()
#         total_samples += labels.size(0)

#         # keep for detailed metrics
#         all_preds.append(preds.cpu())
#         all_labels.append(labels.cpu())
# all_preds = torch.cat(all_preds)
# all_labels = torch.cat(all_labels)
# evaluate(all_labels,all_preds)